# Desafío Data Scientist MRNF: Análisis Exploratorio de Datos (EDA)
**Área:** Modelos de Riesgo No Financiero (MRNF)  
**Objetivo:** Detección de fraude en transacciones con tarjeta maximizando la utilidad económica neta del emisor.  
**Entorno de Ejecución:** `conda activate test-bci` | Python 3.11  
**Estándares:** Restricción estricta Out-of-Time (OOT) a Train/Val (2026-03-01 a 2026-04-30), visualización interactiva Plotly (`plotly_white`), procesamiento de alta velocidad con Polars.

---

### Matriz de Impacto Económico del Negocio
El desempeño del modelo y el análisis exploratorio se orientan a cuantificar la función de utilidad financiera:
$$\text{Ganancia Neta} = (120 \times \text{VP}) - (30 \times \text{FP}) - (180 \times \text{FN}) + (0 \times \text{VN})$$

- **Verdadero Positivo (VP):** +US$ 120 (fraude prevenido, neto de gestión de alerta).
- **Falso Positivo (FP):** -US$ 30 (costo de fricción con cliente y call center).
- **Falso Negativo (FN):** -US$ 180 (fraude consumado, contracargo y reposición de tarjeta).
- **Verdadero Negativo (VN):** US$ 0 (transacción regular sin fricción).


## 1. Configuración del Entorno e Ingesta de Datos


In [ ]:
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import display

# Detección robusta de la raíz del proyecto (funciona tanto desde raíz como desde /notebooks)
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import polars as pl

from src.config import (
    MCC_DESCRIPTIONS,
    PRESENTATION_DIR,
    RAW_DATA_PATH,
)
from src.utils.plots import (
    plot_amount_distribution,
    plot_channel_entry_matrix,
    plot_conditional_missingness,
    plot_geographic_mismatch_risk,
    plot_hourly_seasonality,
    plot_mcc_risk,
    save_interactive_plot,
)

FIGURES_DIR = PRESENTATION_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ruta raíz del proyecto: {PROJECT_ROOT}")
print(f"Ruta datos crudos: {RAW_DATA_PATH}")
print(f"Directorio de figuras interactivas: {FIGURES_DIR}")

### Carga de Datos y Filtrado Ciego de Test Out-of-Time (OOT)
Para garantizar **cero fuga de información (data leakage)** y evitar el sesgo de confirmación durante la formulación de hipótesis, filtramos estrictamente el período de **Entrenamiento y Validación** (`2026-03-01` a `2026-04-30 23:59:59`). El período de mayo de 2026 permanecerá totalmente ciego hasta la evaluación final del modelo.


In [ ]:
# Lectura optimizada con Polars
raw_df = pl.read_csv(RAW_DATA_PATH, try_parse_dates=False)
raw_df = raw_df.with_columns(pl.col("timestamp").str.to_datetime())

total_raw_records = raw_df.height
print(f"Total registros dataset crudo: {total_raw_records:,}")

# Aislamiento temporal estricto (Train + Val: Marzo y Abril 2026)
cutoff_date = datetime(2026, 5, 1)
df_train_val = raw_df.filter(pl.col("timestamp") < cutoff_date)
train_val_records = df_train_val.height

total_fraud = df_train_val.filter(pl.col("is_fraud") == 1).height
fraud_rate = (total_fraud / train_val_records) * 100

print(
    f"Registros en ventana Train/Val: {train_val_records:,} ({(train_val_records / total_raw_records):.1%})"
)
print(f"Casos de fraude en Train/Val: {total_fraud:,}")
print(f"Tasa de fraude global Train/Val: {fraud_rate:.4f}%")

---
## 2. Dimensión 1: Integridad y Valores Faltantes Condicionales
En el dataset general, columnas como `device_id`, `ip_address` e `ip_country` presentan aproximadamente un **59% de nulos**.
**Hipótesis de Negocio:** La ausencia de datos no es un error de recolección ni datos perdidos aleatoriamente (MCAR), sino una propiedad inherente del canal transaccional (MAR/MNAR). En canales físicos (`POS` y `ATM`), no interviene un navegador, app ni dirección IP.


In [ ]:
# Agregación de nulos por canal en Polars
missing_summary = []
for ch in df_train_val["channel"].unique().to_list():
    sub = df_train_val.filter(pl.col("channel") == ch)
    n_sub = sub.height
    for col in ["device_id", "ip_address", "ip_country"]:
        null_count = sub.filter(pl.col(col).is_null()).height
        null_pct = (null_count / n_sub) * 100
        missing_summary.append(
            {
                "channel": ch,
                "feature": col,
                "null_count": null_count,
                "null_pct": null_pct,
            }
        )

df_missing_pd = pd.DataFrame(missing_summary)

fig_missing = plot_conditional_missingness(df_missing_pd)
save_interactive_plot(fig_missing, "01_nulos_condicionales", FIGURES_DIR)
fig_missing.show()

**Conclusión Dimensión 1:**
- En `POS` y `ATM`, el porcentaje de nulos en `device_id`, `ip_address` e `ip_country` es exactamente el **100%**.
- En `ECOMMERCE` y `APP`, la presencia de atributos digitales es **100% completa**.
- **Implicancia para Feature Engineering:** No se debe imputar con valores arbitrarios (moda/media); las variables de dispositivo e IP deben tratarse con flags de disponibilidad de canal y enriquecerse solo cuando el canal sea digital.


---
## 3. Dimensión 2: Análisis de Montos de Transacción (`amount`)
Analizamos si el fraude tiende a concentrarse en montos sustancialmente mayores o si existen patrones discretos (como retiros ATM).


In [ ]:
# Estadísticas descriptivas por clase
amount_stats = (
    df_train_val.group_by("is_fraud")
    .agg(
        [
            pl.count("amount").alias("transacciones"),
            pl.mean("amount").alias("media"),
            pl.median("amount").alias("mediana"),
            pl.col("amount").quantile(0.25).alias("p25"),
            pl.col("amount").quantile(0.75).alias("p75"),
            pl.col("amount").quantile(0.95).alias("p95"),
            pl.col("amount").quantile(0.99).alias("p99"),
            pl.max("amount").alias("maximo"),
        ]
    )
    .to_pandas()
)

display(amount_stats)

# Visualización comparativa en escala logarítmica
sample_amounts = (
    df_train_val.select(["amount", "is_fraud"])
    .sample(n=min(50000, train_val_records), seed=42)
    .to_pandas()
)
fig_amount = plot_amount_distribution(sample_amounts)
save_interactive_plot(fig_amount, "02_distribucion_montos", FIGURES_DIR)
fig_amount.show()

**Conclusión Dimensión 2:**
- El monto promedio en transacciones fraudulentas es considerablemente superior a las legítimas (con alta concentración en percentiles superiores $p95+$).
- La distribución presenta una fuerte asimetría positiva (*right-skewed*), recomendando transformaciones monotónicas tipo `log1p(amount)` o estandarización robusta (`RobustScaler`).
- En ATM, los retiros son estrictamente múltiplos de 10 USD, lo cual debe preservarse como regla sintáctica del canal.


---
## 4. Dimensión 3: Estacionalidad Temporal y Comportamiento Circadiano
Examinamos la distribución horaria de las transacciones y la tasa de fraude correspondiente para detectar ventanas críticas de vulnerabilidad.


In [ ]:
# Agregación por hora del día con Polars
hourly_summary = (
    df_train_val.with_columns(pl.col("timestamp").dt.hour().alias("hour"))
    .group_by("hour")
    .agg(
        [
            pl.count("transaction_id").alias("total_tx"),
            pl.col("is_fraud").sum().alias("fraud_count"),
            (pl.col("is_fraud").mean() * 100).alias("fraud_rate_pct"),
        ]
    )
    .sort("hour")
    .to_pandas()
)

fig_hourly = plot_hourly_seasonality(hourly_summary)
save_interactive_plot(fig_hourly, "03_estacionalidad_horaria", FIGURES_DIR)
fig_hourly.show()

**Conclusión Dimensión 3:**
- Existe una clara asimetría horaria: mientras que el volumen total transaccional cae drásticamente durante la madrugada (01:00 a 06:00), la **tasa de fraude se dispara significativamente** en este período (menos vigilancia y mayores ataques automatizados).
- Esta periodicidad justifica de manera concluyente la utilización de nuestro `CyclicalTimeTransformer` trigonométrico (seno y coseno de hora y día).


---
## 5. Dimensión 4: Riesgo por Rubro Comercial (MCC), Canales y Entry Mode
Cruzamos los códigos de rubro (`mcc`) con sus descripciones comerciales y analizamos la vulnerabilidad del modo de entrada.


In [ ]:
# Riesgo por MCC
mcc_summary = (
    df_train_val.group_by("mcc")
    .agg(
        [
            pl.count("transaction_id").alias("total_tx"),
            pl.col("is_fraud").sum().alias("fraud_count"),
            (pl.col("is_fraud").mean() * 100).alias("fraud_rate_pct"),
        ]
    )
    .to_pandas()
)

mcc_summary["description"] = (
    mcc_summary["mcc"].map(MCC_DESCRIPTIONS).fillna("Desconocido")
)
fig_mcc = plot_mcc_risk(mcc_summary, top_n=15)
save_interactive_plot(fig_mcc, "04_riesgo_mcc", FIGURES_DIR)
fig_mcc.show()

# Matriz Canal vs Entry Mode
matrix_summary = (
    df_train_val.group_by(["channel", "entry_mode"])
    .agg(
        [
            pl.count("transaction_id").alias("total_tx"),
            pl.col("is_fraud").sum().alias("fraud_count"),
            (pl.col("is_fraud").mean() * 100).alias("fraud_rate_pct"),
        ]
    )
    .to_pandas()
)

fig_matrix = plot_channel_entry_matrix(matrix_summary)
save_interactive_plot(fig_matrix, "05_matriz_canal_entrada", FIGURES_DIR)
fig_matrix.show()

**Conclusión Dimensión 4:**
- **MCCs Críticos:** Los rubros con mayor tasa de fraude son `7995 (Apuestas/Casinos)`, `6051 (Cuasi-Efectivo/Cripto)`, `5815 (Bienes Digitales)` y `5732 (Electrónica)`.
- **Modos de Entrada Vulnerables:** Dentro de `POS`, el modo `swipe` (banda magnética) presenta un riesgo de fraude significativamente mayor que `chip` o `contactless`, indicativo de clonación de tarjetas.


---
## 6. Dimensión 5: Huella Digital y Descalce Geográfico
Para canales no presenciales (`ECOMMERCE` y `APP`), evaluamos la discrepancia entre el país del comercio (`merchant_country`) y el país de la IP (`ip_country`), así como la multiplicidad de tarjetas por dispositivo.


In [ ]:
# Análisis de Descalce Geográfico en Canales Digitales
digital_tx = df_train_val.filter(pl.col("channel").is_in(["ECOMMERCE", "APP"]))

geo_analysis = (
    digital_tx.with_columns(
        pl.when(pl.col("merchant_country") == pl.col("ip_country"))
        .then(pl.lit("Coincide (Mismo País)"))
        .otherwise(pl.lit("Descalce (País Distinto)"))
        .alias("mismatch_label")
    )
    .group_by("mismatch_label")
    .agg(
        [
            pl.count("transaction_id").alias("total_tx"),
            pl.col("is_fraud").sum().alias("fraud_count"),
            (pl.col("is_fraud").mean() * 100).alias("fraud_rate_pct"),
        ]
    )
    .to_pandas()
)

fig_geo = plot_geographic_mismatch_risk(geo_analysis)
save_interactive_plot(fig_geo, "06_descalce_geografico", FIGURES_DIR)
fig_geo.show()

# Multiplicidad de tarjetas por dispositivo en canales digitales
device_card_multiplicity = (
    digital_tx.filter(pl.col("device_id").is_not_null())
    .group_by("device_id")
    .agg(
        [
            pl.n_unique("card_id").alias("cards_per_device"),
            pl.count("transaction_id").alias("tx_per_device"),
            pl.col("is_fraud").max().alias("has_fraud"),
        ]
    )
    .to_pandas()
)

multi_card_devices = device_card_multiplicity[
    device_card_multiplicity["cards_per_device"] > 1
]
print(f"Total dispositivos únicos: {len(device_card_multiplicity):,}")
print(f"Dispositivos con múltiples tarjetas: {len(multi_card_devices):,}")
if len(multi_card_devices) > 0:
    fraud_rate_multicard = multi_card_devices["has_fraud"].mean() * 100
    print(f"Tasa de fraude en dispositivos multitarjeta: {fraud_rate_multicard:.2f}%")

**Conclusión Dimensión 5:**
- Cuando el país del comercio y el país de la IP difieren, la tasa de fraude se multiplica sustancialmente.
- Los dispositivos (`device_id`) o IPs asociados a más de una tarjeta bancaria muestran una propensión a fraude muy superior a la media, revelando ataques organizados o uso de credenciales robadas.


---
## 7. Síntesis Ejecutiva y Blueprint para Feature Engineering (Fase 3)

A partir de los hallazgos cuantitativos del EDA, se definen las siguientes familias de variables para enriquecer el modelo en la Fase 3:

| Dimensión | Hallazgo EDA | Variable a Construir (Feature Engineering) |
|---|---|---|
| **Velocidad** | Ráfagas de compras en cortos periodos antes del bloqueo | Conteos y montos acumulados por tarjeta en ventanas de 1h, 24h y 7d (`card_tx_count_1h`, `card_amount_sum_24h`) |
| **Comportamiento Histórico** | Compras anómalas respecto al perfil habitual | Ratio del monto actual respecto a la media móvil histórica de la tarjeta (`amount / hist_card_mean_amount`) |
| **Geografía** | Alto riesgo en transacciones transfronterizas | Flag binario de descalce país comercio vs IP (`is_country_mismatch`), tarjeta fuera de país base |
| **Huella Digital** | Múltiples tarjetas por dispositivo e IP | Conteo de tarjetas distintas vistas por `device_id` y por `ip_address` en ventanas móviles |
| **MCC y Canales** | Fuerte concentración en rubros vulnerables y modo `swipe` | Codificación target regularizada (Out-of-Fold) para `mcc`, flags de alto riesgo (`is_crypto_betting_mcc`), vulnerabilidad de `entry_mode` |
| **Temporalidad** | Disparo de fraude en horas de la madrugada | Componentes trigonométricos continuos (`hour_sin`, `hour_cos`, `day_sin`, `day_cos`, `is_weekend`) |

Todas las figuras interactivas han sido exportadas a la carpeta `presentation/figures/` para su incorporación en la presentación ejecutiva.
